1. metareact

In [1]:
import pandas as pd
from rdkit import Chem
def get_reactive_atom_indices(smiles):
    # 解析 SMILES
    mol = Chem.MolFromSmiles(smiles)
    
    # 获取标记为反应位点的原子
    reactive_atoms = []
    for atom in mol.GetAtoms():
        # 检查是否带有反应位点标记
        if atom.HasProp('molAtomMapNumber'):
            reactive_atoms.append(atom.GetIdx())  # 获取原子序号
    
    return reactive_atoms

from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize

def standardize_smiles_with_atom_map(smiles: str) -> str:
    try:
        # 将 SMILES 转换为分子对象
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            raise ValueError("Invalid SMILES string.")
        
        # 提取位点信息（Atom Maps）
        atom_map = {atom.GetIdx(): atom.GetAtomMapNum() for atom in mol.GetAtoms() if atom.GetAtomMapNum() > 0}
        
        # 标准化分子（使用 Standardizer）
        uncharger = rdMolStandardize.Uncharger()  # 去质子化
        mol = uncharger.uncharge(mol)
        
        # 去除多余氢原子
        mol = Chem.RemoveHs(mol)
        
        # 恢复位点信息（Atom Maps）
        for idx, map_num in atom_map.items():
            mol.GetAtomWithIdx(idx).SetAtomMapNum(map_num)
        
        # 返回标准化后的 SMILES
        standardized_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)
        return standardized_smiles
    except Exception as e:
        return f"Error: {e}"

def df_col_getatomindices(dfname,col_old,flat=False):
    preds = dfname[col_old].to_list()
    preds = [i.split('|') for i in preds]
    try:
        preds = [list(map(standardize_smiles_with_atom_map,i)) for i in preds]
        ranks = [list(map(get_reactive_atom_indices,i)) for i in preds]
    except:
        ranks = []
        for k in preds:
            preds_part = []
            for x in k:
                try:
                    x = standardize_smiles_with_atom_map(x)
                    x_new = get_reactive_atom_indices(x)
                except:
                    x_new = []
                preds_part.append(x_new)
            ranks.append(preds_part)
    if flat == True:
        ranks_new = []
        for i in ranks:
            list_part = []
            for j in i:
                list_part = list_part + j
            ranks_new.append(list_part)
        return ranks_new
    else:
        return ranks
    
def canonicalsmiles(smi):
    mol = Chem.MolFromSmiles(smi)
    smi_new = Chem.MolToSmiles(mol)
    return smi_new

In [2]:
def assign_ranks(atom_scores):
    # 按分数降序排序，并获取排序后的原子序号
    sorted_atoms = sorted(atom_scores.items(), key=lambda x: x[1], reverse=True)
    
    # 创建一个字典，保存原子序号对应的排名
    rank_dict = {}
    for rank, (atom, _) in enumerate(sorted_atoms):
        rank_dict[atom] = rank + 1
    
    # 保持原子序号从小到大的顺序，并更新为对应的排名
    ranked_atoms = {atom: rank_dict[atom] for atom in sorted(atom_scores)}
    
    return ranked_atoms

In [4]:
df = pd.read_csv('pred_results_drugbank_has_cyp_enzyme.csv')


In [6]:
site_truth = df_col_getatomindices(df,'site_truth',flat=True) 
predict_site = df_col_getatomindices(df,'site_predict') 
df['site_truth_atomidx'] = site_truth
df['predict_site_atomidx'] = predict_site
df['sub_enz'] = df['substrate']+'|' + df['Enzyme']
df

[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Running Uncharger
[11:38:11] Run

,Unnamed: 0,substrate,Enzyme,truth,site_truth,site_predict,site_truth_atomidx,predict_site_atomidx,sub_enz
0,0,CC(C)(C)c1cc(C(C)(C)C)c(NC(=O)c2c[nH]c3ccccc3c...,CYP3A4,CC(C)(C)c1cc(C(C)(C)C(=O)O)c(O)cc1NC(=O)c1c[nH...,CC(C)(C)c1cc(C(C)(C)[CH3:1])c(O)cc1NC(=O)c1c[n...,CC(C)(C)c1cc(C(C)(C)[CH3:1])c(NC(=O)c2c[nH]c3c...,[10],"[[10], [10], [21], [20], [23], [10, 12], [18, ...",CC(C)(C)c1cc(C(C)(C)C)c(NC(=O)c2c[nH]c3ccccc3c...
1,1,CC(C)(C)c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OCC...,CYP3A4,CC(C)(CO)c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OC...,CC(C)(c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OCCN5...,CC(C)(c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OCCN5...,[39],"[[39], [32], [27, 30, 31, 32], [30, 31], [30, ...",CC(C)(C)c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OCC...
2,2,CC1(C)CC(=O)N(CCCCN2CCN(c3ncccn3)CC2)C(=O)C1,CYP3A4,CC1(C)CC(=O)N(CCCCN2CCN(c3ncccn3)CC2)C(=O)C1O|...,CC1(C)CC(=O)N(CCCCN2CCN(c3ncccn3)CC2)C(=O)[CH2...,CC1(C)CC(=O)N(CCCC[N:1]2CCN(c3ncccn3)CC2)C(=O)...,"[25, 11]","[[11], [18], [2, 11], [11], [25], [2], [14], [...",CC1(C)CC(=O)N(CCCCN2CCN(c3ncccn3)CC2)C(=O)C1|C...
3,3,CC1=C(/C=C/C(C)=C/C=C/C(C)=C/C(=O)O)C(C)(C)CCC1,CYP2C9,CC1=C(/C=C/C(C)=C\C=C\C(C)=C\C(=O)O)C(C)(C)CCC1,CC1=C(/C=C/C(C)=C/C=C/C(C)=C/C(=O)O)C(C)(C)CCC1,CC(/C=C/C1=C([CH3:1])CCCC1(C)C)=C\C=C\C(C)=C\C...,[],"[[6], [21], [4, 5], [20], [21], [19], [20], []...",CC1=C(/C=C/C(C)=C/C=C/C(C)=C/C(=O)O)C(C)(C)CCC...
4,4,CC1=C(/C=C/C(C)=C/C=C/C(C)=C/C(=O)O)C(C)(C)CCC1,CYP3A4,CC1=C(/C=C/C(C)=C/C=C/C(C)=C\C(=O)O)C(C)(C)CCC...,CC1=C(/C=C/C(C)=C/C=C/C(C)=C/C(=O)O)C(C)(C)CCC1,CC(/C=C/C1=C([CH3:1])CCCC1(C)C)=C\C=C\C(C)=C\C...,[],"[[6], [21], [4, 5], [20], [21], [19], [20], []...",CC1=C(/C=C/C(C)=C/C=C/C(C)=C/C(=O)O)C(C)(C)CCC...
5,5,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...,CYP1A2,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc([O:1]C)c(C...,[],"[[21], [4], [20, 21, 29, 30], [17, 18, 19, 20,...",CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...
6,6,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...,CYP2D6,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc([O:1]C)c(C...,[],"[[21], [4], [20, 21, 29, 30], [17, 18, 19, 20,...",CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...
7,7,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...,CYP3A4,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc([O:1]C)c(C...,[],"[[21], [4], [20, 21, 29, 30], [17, 18, 19, 20,...",CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...
8,8,CCC(C)n1ncn(-c2ccc(N3CCN(c4ccc(OC[C@H]5CO[C@](...,CYP3A4,CC(O)C(C)n1ncn(-c2ccc(N3CCN(c4ccc(OC[C@@H]5CO[...,CC(n1ncn(-c2ccc(N3CCN(c4ccc(OC[C@H]5CO[C@](Cn6...,CCC(C)[n:1]1ncn(-c2ccc(N3CCN(c4ccc(OC[C@H]5CO[...,[47],"[[4], [47], [22, 40], [47], [4], [48], [47], [...",CCC(C)n1ncn(-c2ccc(N3CCN(c4ccc(OC[C@H]5CO[C@](...
9,9,CCCc1nc(C)c2c(=O)nc(-c3cc(S(=O)(=O)N4CCN(CC)CC...,CYP3A4,CCCc1nc(C)c2c(=O)nc(-c3cc(S(=O)(=O)N4CCNCC4)cc...,CCCc1nc(C)c2c(=O)nc(-c3cc(S(=O)(=O)N4CC[N:1](C...,CCCc1nc(C)c2c(=O)nc(-c3cc(S(=O)(=O)N4CC[N:1](C...,[21],"[[21], [24], [21], [22], [25], [21, 24], [21],...",CCCc1nc(C)c2c(=O)nc(-c3cc(S(=O)(=O)N4CCN(CC)CC...


In [7]:
df.to_pickle('./predict_site_123_has_enzyme.pickle')

2. smartcyp

In [8]:
df_smartcyp = pd.read_pickle('/home/datahouse1/wangyitian/MetaPred/30_main_products/save_code/123/smartcyp_results_drugbank.pickle')
substrates = df_smartcyp['substrates'].to_list()
substrates = list(map(canonicalsmiles,substrates))
df_smartcyp['substrates'] = substrates
df_smartcyp['sub_enz'] = df_smartcyp['substrates'] + '|CYP' + df_smartcyp['enzyme']
df_smartcyp

,substrates,enzyme,scores,sub_enz
0,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)O,3A4,"{0: 3, 1: 21, 2: 10, 3: 1, 4: 20, 5: 17, 6: 16...",CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)...
1,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)O,2D6,"{0: 6, 1: 21, 2: 13, 3: 3, 4: 20, 5: 17, 6: 16...",CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)...
2,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)O,2C9,"{0: 5, 1: 18, 2: 9, 3: 2, 4: 17, 5: 15, 6: 14,...",CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)...
3,CC(=O)Nc1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2,3A4,"{0: 3, 1: 14, 2: 13, 3: 11, 4: 18, 5: 5, 6: 10...",CC(=O)Nc1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2|CYP3A4
4,CC(=O)Nc1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2,2D6,"{0: 2, 1: 14, 2: 13, 3: 9, 4: 18, 5: 7, 6: 10,...",CC(=O)Nc1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2|CYP2D6
...,...,...,...,...
364,Oc1ccc(COCC[N+]23CCC(C(O)(c4ccccc4)c4ccccc4)(C...,2D6,"{0: 12, 1: 13, 2: 6, 3: 11, 4: 16, 5: 4, 6: 19...",Oc1ccc(COCC[N+]23CCC(C(O)(c4ccccc4)c4ccccc4)(C...
365,Oc1ccc(COCC[N+]23CCC(C(O)(c4ccccc4)c4ccccc4)(C...,2C9,"{0: 12, 1: 13, 2: 6, 3: 11, 4: 16, 5: 3, 6: 19...",Oc1ccc(COCC[N+]23CCC(C(O)(c4ccccc4)c4ccccc4)(C...
366,c1cnc(N2CCNCC2)nc1,3A4,"{0: 6, 1: 7, 2: 5, 3: 8, 4: 4, 5: 1, 6: 3, 7: ...",c1cnc(N2CCNCC2)nc1|CYP3A4
367,c1cnc(N2CCNCC2)nc1,2D6,"{0: 1, 1: 5, 2: 4, 3: 8, 4: 7, 5: 2, 6: 6, 7: ...",c1cnc(N2CCNCC2)nc1|CYP2D6


In [9]:
scores_smartcyp = df_smartcyp['scores'].to_list()
smartcyp_sorted_atoms = [[k for k, v in sorted(i.items(), key=lambda item: item[1])] for i in scores_smartcyp]
df_smartcyp['rank'] = smartcyp_sorted_atoms
df_smartcyp

,substrates,enzyme,scores,sub_enz,rank
0,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)O,3A4,"{0: 3, 1: 21, 2: 10, 3: 1, 4: 20, 5: 17, 6: 16...",CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)...,"[3, 18, 0, 9, 8, 10, 19, 16, 15, 2, 14, 7, 12,..."
1,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)O,2D6,"{0: 6, 1: 21, 2: 13, 3: 3, 4: 20, 5: 17, 6: 16...",CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)...,"[9, 18, 3, 8, 10, 0, 19, 7, 12, 13, 15, 14, 16..."
2,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)O,2C9,"{0: 5, 1: 18, 2: 9, 3: 2, 4: 17, 5: 15, 6: 14,...",CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)...,"[9, 3, 8, 10, 0, 7, 12, 13, 18, 2, 14, 15, 19,..."
3,CC(=O)Nc1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2,3A4,"{0: 3, 1: 14, 2: 13, 3: 11, 4: 18, 5: 5, 6: 10...",CC(=O)Nc1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2|CYP3A4,"[19, 18, 0, 14, 5, 12, 9, 13, 15, 6, 3, 22, 2,..."
4,CC(=O)Nc1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2,2D6,"{0: 2, 1: 14, 2: 13, 3: 9, 4: 18, 5: 7, 6: 10,...",CC(=O)Nc1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2|CYP2D6,"[19, 0, 14, 18, 13, 15, 5, 12, 3, 6, 9, 22, 2,..."
...,...,...,...,...,...
364,Oc1ccc(COCC[N+]23CCC(C(O)(c4ccccc4)c4ccccc4)(C...,2D6,"{0: 12, 1: 13, 2: 6, 3: 11, 4: 16, 5: 4, 6: 19...",Oc1ccc(COCC[N+]23CCC(C(O)(c4ccccc4)c4ccccc4)(C...,"[8, 9, 18, 24, 5, 7, 2, 32, 17, 19, 23, 25, 10..."
365,Oc1ccc(COCC[N+]23CCC(C(O)(c4ccccc4)c4ccccc4)(C...,2C9,"{0: 12, 1: 13, 2: 6, 3: 11, 4: 16, 5: 3, 6: 19...",Oc1ccc(COCC[N+]23CCC(C(O)(c4ccccc4)c4ccccc4)(C...,"[8, 9, 5, 18, 24, 7, 2, 32, 10, 28, 30, 17, 19..."
366,c1cnc(N2CCNCC2)nc1,3A4,"{0: 6, 1: 7, 2: 5, 3: 8, 4: 4, 5: 1, 6: 3, 7: ...",c1cnc(N2CCNCC2)nc1|CYP3A4,"[5, 9, 7, 6, 8, 4, 2, 10, 0, 1, 11, 3]"
367,c1cnc(N2CCNCC2)nc1,2D6,"{0: 1, 1: 5, 2: 4, 3: 8, 4: 7, 5: 2, 6: 6, 7: ...",c1cnc(N2CCNCC2)nc1|CYP2D6,"[0, 5, 9, 7, 2, 10, 1, 11, 6, 8, 4, 3]"


In [10]:
dict_smartcyp_sub2rank = dict(zip(df_smartcyp['sub_enz'].to_list(),smartcyp_sorted_atoms))
dict_smartcyp_sub2rank

{'CC(/C=C/C12OC1(C)CCCC2(C)C)=C\\C=C\\C(C)=C\\C(=O)O|CYP3A4': [3,
  18,
  0,
  9,
  8,
  10,
  19,
  16,
  15,
  2,
  14,
  7,
  12,
  13,
  21,
  22,
  20,
  6,
  5,
  11,
  17,
  4,
  1],
 'CC(/C=C/C12OC1(C)CCCC2(C)C)=C\\C=C\\C(C)=C\\C(=O)O|CYP2D6': [9,
  18,
  3,
  8,
  10,
  0,
  19,
  7,
  12,
  13,
  15,
  14,
  16,
  2,
  21,
  22,
  20,
  6,
  5,
  11,
  17,
  4,
  1],
 'CC(/C=C/C12OC1(C)CCCC2(C)C)=C\\C=C\\C(C)=C\\C(=O)O|CYP2C9': [9,
  3,
  8,
  10,
  0,
  7,
  12,
  13,
  18,
  2,
  14,
  15,
  19,
  16,
  6,
  5,
  11,
  4,
  1,
  22,
  20,
  17,
  21],
 'CC(=O)Nc1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2|CYP3A4': [19,
  18,
  0,
  14,
  5,
  12,
  9,
  13,
  15,
  6,
  3,
  22,
  2,
  1,
  21,
  20,
  16,
  4,
  11,
  7,
  10,
  8],
 'CC(=O)Nc1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2|CYP2D6': [19,
  0,
  14,
  18,
  13,
  15,
  5,
  12,
  3,
  6,
  9,
  22,
  2,
  1,
  21,
  20,
  16,
  4,
  11,
  7,
  10,
  8],
 'CC(=O)Nc1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2|CYP2C9': [19,
  0,
  18,
  14,
  13,

In [11]:
df_smartcyp.to_pickle('./smartcyp_cyp_123.pickle')

3. somp

In [12]:
df_somp = pd.read_pickle('/home/datahouse1/wangyitian/MetaPred/30_main_products/save_code/123/somp_results_drugbank.pickle')
df_somp = df_somp[df_somp['enzyme']!='UGT']
substrates_somp = df_somp['substrates'].to_list()
substrates_somp = list(map(canonicalsmiles,substrates_somp))
df_somp['substrates'] = substrates_somp
df_somp['sub_enz'] = df_somp['substrates'] + '|CYP' + df_somp['enzyme']
scores_somp = df_somp['scores'].to_list()
somp_sorted_atoms = [[k for k, v in sorted(i.items(), key=lambda item: item[1])] for i in scores_somp]
df_somp['rank'] = somp_sorted_atoms
df_somp

,substrates,enzyme,scores,sub_enz,rank
0,C[C@H](N)Cc1ccccc1,2C9,"{0: 8, 1: 1, 2: 3, 3: 7, 4: 6, 5: 11, 6: 9, 7:...",C[C@H](N)Cc1ccccc1|CYP2C9,"[1, 8, 2, 7, 9, 4, 3, 0, 6, 10, 5]"
1,N#Cc1ccc2c(c1)COC2(CCCN)c1ccc(F)cc1,2D6,"{0: 11, 1: 8, 2: 17, 3: 6, 4: 7, 5: 20, 6: 19,...",N#Cc1ccc2c(c1)COC2(CCCN)c1ccc(F)cc1|CYP2D6,"[17, 20, 13, 16, 21, 3, 4, 1, 18, 8, 0, 19, 14..."
2,c1cnc(N2CCNCC2)nc1,1A2,"{0: 1, 1: 7, 2: 9, 3: 12, 4: 11, 5: 5, 6: 3, 7...",c1cnc(N2CCNCC2)nc1|CYP1A2,"[0, 7, 6, 8, 5, 9, 1, 11, 2, 10, 4, 3]"
3,Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...,2D6,"{0: 2, 1: 26, 2: 17, 3: 19, 4: 22, 5: 21, 6: 1...",Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...,"[27, 0, 20, 9, 14, 16, 17, 28, 15, 7, 21, 26, ..."
4,O=C1Nc2ccc([N+](=O)[O-])cc2C(c2ccccc2Cl)=NC1O,2C9,"{0: 13, 1: 23, 2: 19, 3: 18, 4: 10, 5: 11, 6: ...",O=C1Nc2ccc([N+](=O)[O-])cc2C(c2ccccc2Cl)=NC1O|...,"[21, 7, 16, 8, 9, 15, 22, 17, 20, 4, 5, 14, 0,..."
...,...,...,...,...,...
694,CC1=C(/C=C/C(C)=C\C=C\C(C)=C\C(=O)O)C(C)(C)CCC1,2D6,"{0: 2, 1: 16, 2: 20, 3: 9, 4: 10, 5: 5, 6: 11,...",CC1=C(/C=C/C(C)=C\C=C\C(C)=C\C(=O)O)C(C)(C)CCC...,"[21, 0, 20, 19, 5, 8, 17, 18, 3, 4, 6, 11, 7, ..."
695,O=C1CC(O)CC(/C=C/c2c(C3CC3)nc3ccccc3c2-c2ccc(F...,2C9,"{0: 10, 1: 20, 2: 1, 3: 5, 4: 24, 5: 6, 6: 18,...",O=C1CC(O)CC(/C=C/c2c(C3CC3)nc3ccccc3c2-c2ccc(F...,"[2, 20, 7, 21, 3, 5, 12, 17, 22, 0, 27, 28, 19..."
696,CS(=O)(=O)c1ccc(C(=O)Nc2ccc(Cl)c(-c3ccccn3)c2)...,3A4,"{0: 18, 1: 1, 2: 16, 3: 17, 4: 27, 5: 11, 6: 1...",CS(=O)(=O)c1ccc(C(=O)Nc2ccc(Cl)c(-c3ccccn3)c2)...,"[1, 20, 12, 22, 13, 19, 23, 18, 21, 26, 5, 6, ..."
698,O=C1CN=C(c2ccccc2F)c2cc(Cl)ccc2N1CC(F)(F)F,3A4,"{0: 14, 1: 21, 2: 1, 3: 13, 4: 24, 5: 23, 6: 6...",O=C1CN=C(c2ccccc2F)c2cc(Cl)ccc2N1CC(F)(F)F|CYP3A4,"[2, 20, 16, 8, 7, 6, 17, 13, 9, 22, 23, 24, 3,..."


In [13]:
df_somp.to_pickle('./somp_cyp_123.pickle')

In [14]:
dict_somp_sub2rank = dict(zip(df_somp['sub_enz'].to_list(),somp_sorted_atoms))
dict_somp_sub2rank

{'C[C@H](N)Cc1ccccc1|CYP2C9': [1, 8, 2, 7, 9, 4, 3, 0, 6, 10, 5],
 'N#Cc1ccc2c(c1)COC2(CCCN)c1ccc(F)cc1|CYP2D6': [17,
  20,
  13,
  16,
  21,
  3,
  4,
  1,
  18,
  8,
  0,
  19,
  14,
  12,
  7,
  11,
  2,
  15,
  6,
  5,
  10,
  9],
 'c1cnc(N2CCNCC2)nc1|CYP1A2': [0, 7, 6, 8, 5, 9, 1, 11, 2, 10, 4, 3],
 'Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(CCO)CC2)n1|CYP2D6': [27,
  0,
  20,
  9,
  14,
  16,
  17,
  28,
  15,
  7,
  21,
  26,
  25,
  30,
  24,
  31,
  2,
  6,
  3,
  10,
  5,
  4,
  8,
  13,
  19,
  1,
  22,
  12,
  18,
  32,
  23,
  11,
  29],
 'O=C1Nc2ccc([N+](=O)[O-])cc2C(c2ccccc2Cl)=NC1O|CYP2C9': [21,
  7,
  16,
  8,
  9,
  15,
  22,
  17,
  20,
  4,
  5,
  14,
  0,
  19,
  10,
  18,
  6,
  3,
  2,
  13,
  11,
  12,
  1],
 'Cc1c(OCC(F)(F)F)ccnc1C[S@@](=O)c1nc2ccc(O)cc2[nH]1|CYP1A2': [14,
  4,
  20,
  0,
  23,
  19,
  6,
  7,
  8,
  9,
  5,
  21,
  15,
  10,
  11,
  2,
  13,
  17,
  18,
  24,
  22,
  1,
  3,
  16,
  25,
  12],
 'COc1nc2ccc(Br)cc2cc1[C@@H](c1ccccc1)[C@@](O)(

4. 整理结果

In [15]:
substrates_total = df['sub_enz'].to_list()
rank_smartcyp = [dict_smartcyp_sub2rank.get(i,'None') for i in substrates_total]
rank_somp= [dict_somp_sub2rank.get(i,'None') for i in substrates_total]
df['rank_smartcyp'] = rank_smartcyp
df['rank_somp'] = rank_somp
df

,Unnamed: 0,substrate,Enzyme,truth,site_truth,site_predict,site_truth_atomidx,predict_site_atomidx,sub_enz,rank_smartcyp,rank_somp
0,0,CC(C)(C)c1cc(C(C)(C)C)c(NC(=O)c2c[nH]c3ccccc3c...,CYP3A4,CC(C)(C)c1cc(C(C)(C)C(=O)O)c(O)cc1NC(=O)c1c[nH...,CC(C)(C)c1cc(C(C)(C)[CH3:1])c(O)cc1NC(=O)c1c[n...,CC(C)(C)c1cc(C(C)(C)[CH3:1])c(NC(=O)c2c[nH]c3c...,[10],"[[10], [10], [21], [20], [23], [10, 12], [18, ...",CC(C)(C)c1cc(C(C)(C)C)c(NC(=O)c2c[nH]c3ccccc3c...,"[28, 21, 17, 26, 20, 22, 19, 0, 2, 3, 5, 8, 9,...","[0, 2, 3, 8, 9, 10, 21, 20, 19, 26, 7, 22, 1, ..."
1,1,CC(C)(C)c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OCC...,CYP3A4,CC(C)(CO)c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OC...,CC(C)(c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OCCN5...,CC(C)(c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OCCN5...,[39],"[[39], [32], [27, 30, 31, 32], [30, 31], [30, ...",CC(C)(C)c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OCC...,"[28, 32, 26, 27, 29, 31, 25, 34, 20, 33, 22, 5...","[26, 20, 0, 2, 3, 25, 1, 28, 32, 12, 37, 29, 3..."
2,2,CC1(C)CC(=O)N(CCCCN2CCN(c3ncccn3)CC2)C(=O)C1,CYP3A4,CC1(C)CC(=O)N(CCCCN2CCN(c3ncccn3)CC2)C(=O)C1O|...,CC1(C)CC(=O)N(CCCCN2CCN(c3ncccn3)CC2)C(=O)[CH2...,CC1(C)CC(=O)N(CCCC[N:1]2CCN(c3ncccn3)CC2)C(=O)...,"[25, 11]","[[11], [18], [2, 11], [11], [25], [2], [14], [...",CC1(C)CC(=O)N(CCCCN2CCN(c3ncccn3)CC2)C(=O)C1|C...,"[13, 21, 10, 11, 12, 22, 14, 16, 20, 3, 25, 7,...","[10, 3, 25, 18, 7, 11, 12, 22, 13, 21, 0, 2, 9..."
3,3,CC1=C(/C=C/C(C)=C/C=C/C(C)=C/C(=O)O)C(C)(C)CCC1,CYP2C9,CC1=C(/C=C/C(C)=C\C=C\C(C)=C\C(=O)O)C(C)(C)CCC1,CC1=C(/C=C/C(C)=C/C=C/C(C)=C/C(=O)O)C(C)(C)CCC1,CC(/C=C/C1=C([CH3:1])CCCC1(C)C)=C\C=C\C(C)=C\C...,[],"[[6], [21], [4, 5], [20], [21], [19], [20], []...",CC1=C(/C=C/C(C)=C/C=C/C(C)=C/C(=O)O)C(C)(C)CCC...,"[0, 21, 20, 19, 6, 17, 18, 11, 4, 3, 7, 8, 12,...","[21, 0, 2, 1, 19, 20, 17, 18, 8, 5, 6, 11, 3, ..."
4,4,CC1=C(/C=C/C(C)=C/C=C/C(C)=C/C(=O)O)C(C)(C)CCC1,CYP3A4,CC1=C(/C=C/C(C)=C/C=C/C(C)=C\C(=O)O)C(C)(C)CCC...,CC1=C(/C=C/C(C)=C/C=C/C(C)=C/C(=O)O)C(C)(C)CCC1,CC(/C=C/C1=C([CH3:1])CCCC1(C)C)=C\C=C\C(C)=C\C...,[],"[[6], [21], [4, 5], [20], [21], [19], [20], []...",CC1=C(/C=C/C(C)=C/C=C/C(C)=C/C(=O)O)C(C)(C)CCC...,"[0, 21, 11, 6, 20, 19, 12, 9, 3, 8, 4, 7, 17, ...","[21, 0, 2, 1, 19, 20, 8, 3, 17, 18, 4, 5, 6, 1..."
5,5,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...,CYP1A2,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc([O:1]C)c(C...,[],"[[21], [4], [20, 21, 29, 30], [17, 18, 19, 20,...",CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...,None,"[23, 1, 0, 5, 31, 2, 8, 29, 32, 16, 6, 30, 12,..."
6,6,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...,CYP2D6,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc([O:1]C)c(C...,[],"[[21], [4], [20, 21, 29, 30], [17, 18, 19, 20,...",CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...,"[22, 29, 15, 7, 8, 1, 0, 16, 30, 5, 19, 6, 31,...","[23, 1, 0, 5, 2, 30, 8, 12, 16, 29, 32, 19, 31..."
7,7,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...,CYP3A4,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...,CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc([O:1]C)c(C...,[],"[[21], [4], [20, 21, 29, 30], [17, 18, 19, 20,...",CCC(=O)N1CC[C@H](Nc2ncnc3c2CN(c2cnc(OC)c(C(F)(...,"[29, 7, 15, 8, 22, 16, 30, 1, 5, 19, 6, 31, 12...","[5, 32, 23, 0, 16, 1, 30, 2, 29, 31, 8, 4, 6, ..."
8,8,CCC(C)n1ncn(-c2ccc(N3CCN(c4ccc(OC[C@H]5CO[C@](...,CYP3A4,CC(O)C(C)n1ncn(-c2ccc(N3CCN(c4ccc(OC[C@@H]5CO[...,CC(n1ncn(-c2ccc(N3CCN(c4ccc(OC[C@H]5CO[C@](Cn6...,CCC(C)[n:1]1ncn(-c2ccc(N3CCN(c4ccc(OC[C@H]5CO[...,[47],"[[4], [47], [22, 40], [47], [4], [48], [47], [...",CCC(C)n1ncn(-c2ccc(N3CCN(c4ccc(OC[C@H]5CO[C@](...,"[28, 6, 10, 45, 13, 44, 17, 42, 14, 43, 23, 21...","[2, 1, 21, 3, 35, 0, 38, 27, 13, 14, 44, 45, 1..."
9,9,CCCc1nc(C)c2c(=O)nc(-c3cc(S(=O)(=O)N4CCN(CC)CC...,CYP3A4,CCCc1nc(C)c2c(=O)nc(-c3cc(S(=O)(=O)N4CCNCC4)cc...,CCCc1nc(C)c2c(=O)nc(-c3cc(

In [16]:
df.to_pickle('./total_cyp_123.pickle')

In [17]:
def get_unique_numbers(nums, n):
    """
    从列表中提取前 n 个不同的数字。

    参数:
        nums (list): 输入的数字列表。
        n (int): 需要提取的不同数字的数量。

    返回:
        list: 包含前 n 个不同数字的列表。
    """
    unique_nums = []
    seen = set()

    for num in nums:
        if num not in seen:
            unique_nums.append(num)
            seen.add(num)
        if len(unique_nums) == n:
            break

    return unique_nums

In [18]:
df_total = pd.read_pickle('./total_cyp_123.pickle')

ourmodel_results_old = df_total['predict_site_atomidx'].to_list()
ourmodel_results_new = []
for i in ourmodel_results_old:
    list_part = []
    for j in i:
        list_part = list_part + j
    ourmodel_results_new.append(list_part)

true_sites_old = df_total['site_truth_atomidx'].to_list()

def acc_top_n(n,modelresults,true_sites):
    modelresults = [i[:n] for i in modelresults]
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

def acc_top_n_old(n,modelresults,true_sites):
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = get_unique_numbers(modelresult, n)
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

smartcyp_ranks = df_total['rank_smartcyp'].to_list()
smartcyp_top1_acc = acc_top_n(1,smartcyp_ranks,true_sites_old)
smartcyp_top3_acc = acc_top_n(3,smartcyp_ranks,true_sites_old)
smartcyp_top5_acc = acc_top_n(5,smartcyp_ranks,true_sites_old)
smartcyp_topn = []
smartcyp_topn.append(smartcyp_top1_acc)
smartcyp_topn.append(smartcyp_top3_acc)
smartcyp_topn.append(smartcyp_top5_acc)

somp_ranks = df_total['rank_somp'].to_list()
somp_top1_acc = acc_top_n(1,somp_ranks,true_sites_old)
somp_top3_acc = acc_top_n(3,somp_ranks,true_sites_old)
somp_top5_acc = acc_top_n(5,somp_ranks,true_sites_old)
somp_topn = []
somp_topn.append(somp_top1_acc)
somp_topn.append(somp_top3_acc)
somp_topn.append(somp_top5_acc)

ourmodel_top1_acc = acc_top_n_old(1,ourmodel_results_new,true_sites_old)
ourmodel_top3_acc = acc_top_n_old(3,ourmodel_results_new,true_sites_old)
ourmodel_top5_acc = acc_top_n_old(5,ourmodel_results_new,true_sites_old)
ourmodel_topn = []
ourmodel_topn.append(ourmodel_top1_acc)
ourmodel_topn.append(ourmodel_top3_acc)
ourmodel_topn.append(ourmodel_top5_acc)

top 1 accuracy = 0.04
top 3 accuracy = 0.26
top 5 accuracy = 0.4
top 1 accuracy = 0.14
top 3 accuracy = 0.34
top 5 accuracy = 0.46
top 1 accuracy = 0.5
top 3 accuracy = 0.74
top 5 accuracy = 0.86


In [19]:
df = pd.DataFrame()
df['top_n'] = ['top1_recall','top3_recall','top5_recall']
df['smartcyp_topn'] = smartcyp_topn
df['somp_topn'] = somp_topn
df['ourmodel_topn'] = ourmodel_topn
df

,top_n,smartcyp_topn,somp_topn,ourmodel_topn
0,top1_recall,0.04,0.14,0.50
1,top3_recall,0.26,0.34,0.74
2,top5_recall,0.40,0.46,0.86


5. 按照酶整理结果

In [20]:
df_total['Enzyme'].unique()

array(['CYP3A4', 'CYP2C9', 'CYP1A2', 'CYP2D6'], dtype=object)

In [21]:
df_total_cyp1a2 = df_total[df_total['Enzyme'] == 'CYP1A2']

ourmodel_results_old = df_total_cyp1a2['predict_site_atomidx'].to_list()
ourmodel_results_new = []
for i in ourmodel_results_old:
    list_part = []
    for j in i:
        list_part = list_part + j
    ourmodel_results_new.append(list_part)

true_sites_old = df_total_cyp1a2['site_truth_atomidx'].to_list()

def acc_top_n(n,modelresults,true_sites):
    modelresults = [i[:n] for i in modelresults]
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

def acc_top_n_old(n,modelresults,true_sites):
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = get_unique_numbers(modelresult, n)
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

smartcyp_ranks = df_total_cyp1a2['rank_smartcyp'].to_list()
smartcyp_top1_acc = acc_top_n(1,smartcyp_ranks,true_sites_old)
smartcyp_top3_acc = acc_top_n(3,smartcyp_ranks,true_sites_old)
smartcyp_top5_acc = acc_top_n(5,smartcyp_ranks,true_sites_old)
smartcyp_topn = []
smartcyp_topn.append(smartcyp_top1_acc)
smartcyp_topn.append(smartcyp_top3_acc)
smartcyp_topn.append(smartcyp_top5_acc)

somp_ranks = df_total_cyp1a2['rank_somp'].to_list()
somp_top1_acc = acc_top_n(1,somp_ranks,true_sites_old)
somp_top3_acc = acc_top_n(3,somp_ranks,true_sites_old)
somp_top5_acc = acc_top_n(5,somp_ranks,true_sites_old)
somp_topn = []
somp_topn.append(somp_top1_acc)
somp_topn.append(somp_top3_acc)
somp_topn.append(somp_top5_acc)

ourmodel_top1_acc = acc_top_n_old(1,ourmodel_results_new,true_sites_old)
ourmodel_top3_acc = acc_top_n_old(3,ourmodel_results_new,true_sites_old)
ourmodel_top5_acc = acc_top_n_old(5,ourmodel_results_new,true_sites_old)
ourmodel_topn = []
ourmodel_topn.append(ourmodel_top1_acc)
ourmodel_topn.append(ourmodel_top3_acc)
ourmodel_topn.append(ourmodel_top5_acc)
df = pd.DataFrame()
df['top_n'] = ['top1_recall','top3_recall','top5_recall']
df['smartcyp_topn'] = smartcyp_topn
df['somp_topn'] = somp_topn
df['ourmodel_topn'] = ourmodel_topn
df

top 1 accuracy = 0.0
top 3 accuracy = 0.0
top 5 accuracy = 0.0
top 1 accuracy = 0.0
top 3 accuracy = 0.6666666666666666
top 5 accuracy = 0.6666666666666666
top 1 accuracy = 0.3333333333333333
top 3 accuracy = 0.6666666666666666
top 5 accuracy = 0.6666666666666666


,top_n,smartcyp_topn,somp_topn,ourmodel_topn
0,top1_recall,0.0,0.000000,0.333333
1,top3_recall,0.0,0.666667,0.666667
2,top5_recall,0.0,0.666667,0.666667


In [22]:
df_total_cyp2c9 = df_total[df_total['Enzyme'] == 'CYP2C9']

ourmodel_results_old = df_total_cyp2c9['predict_site_atomidx'].to_list()
ourmodel_results_new = []
for i in ourmodel_results_old:
    list_part = []
    for j in i:
        list_part = list_part + j
    ourmodel_results_new.append(list_part)

true_sites_old = df_total_cyp2c9['site_truth_atomidx'].to_list()

def acc_top_n(n,modelresults,true_sites):
    modelresults = [i[:n] for i in modelresults]
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

def acc_top_n_old(n,modelresults,true_sites):
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = get_unique_numbers(modelresult, n)
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

smartcyp_ranks = df_total_cyp2c9['rank_smartcyp'].to_list()
smartcyp_top1_acc = acc_top_n(1,smartcyp_ranks,true_sites_old)
smartcyp_top3_acc = acc_top_n(3,smartcyp_ranks,true_sites_old)
smartcyp_top5_acc = acc_top_n(5,smartcyp_ranks,true_sites_old)
smartcyp_topn = []
smartcyp_topn.append(smartcyp_top1_acc)
smartcyp_topn.append(smartcyp_top3_acc)
smartcyp_topn.append(smartcyp_top5_acc)

somp_ranks = df_total_cyp2c9['rank_somp'].to_list()
somp_top1_acc = acc_top_n(1,somp_ranks,true_sites_old)
somp_top3_acc = acc_top_n(3,somp_ranks,true_sites_old)
somp_top5_acc = acc_top_n(5,somp_ranks,true_sites_old)
somp_topn = []
somp_topn.append(somp_top1_acc)
somp_topn.append(somp_top3_acc)
somp_topn.append(somp_top5_acc)

ourmodel_top1_acc = acc_top_n_old(1,ourmodel_results_new,true_sites_old)
ourmodel_top3_acc = acc_top_n_old(3,ourmodel_results_new,true_sites_old)
ourmodel_top5_acc = acc_top_n_old(5,ourmodel_results_new,true_sites_old)
ourmodel_topn = []
ourmodel_topn.append(ourmodel_top1_acc)
ourmodel_topn.append(ourmodel_top3_acc)
ourmodel_topn.append(ourmodel_top5_acc)

df = pd.DataFrame()
df['top_n'] = ['top1_recall','top3_recall','top5_recall']
df['smartcyp_topn'] = smartcyp_topn
df['somp_topn'] = somp_topn
df['ourmodel_topn'] = ourmodel_topn
df

top 1 accuracy = 0.0
top 3 accuracy = 0.3333333333333333
top 5 accuracy = 0.4444444444444444
top 1 accuracy = 0.1111111111111111
top 3 accuracy = 0.3333333333333333
top 5 accuracy = 0.3333333333333333
top 1 accuracy = 0.3333333333333333
top 3 accuracy = 0.7777777777777778
top 5 accuracy = 0.7777777777777778


,top_n,smartcyp_topn,somp_topn,ourmodel_topn
0,top1_recall,0.000000,0.111111,0.333333
1,top3_recall,0.333333,0.333333,0.777778
2,top5_recall,0.444444,0.333333,0.777778


In [23]:
df_total_cyp2d6 = df_total[df_total['Enzyme'] == 'CYP2D6']

ourmodel_results_old = df_total_cyp2d6['predict_site_atomidx'].to_list()
ourmodel_results_new = []
for i in ourmodel_results_old:
    list_part = []
    for j in i:
        list_part = list_part + j
    ourmodel_results_new.append(list_part)

true_sites_old = df_total_cyp2d6['site_truth_atomidx'].to_list()

def acc_top_n(n,modelresults,true_sites):
    modelresults = [i[:n] for i in modelresults]
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

def acc_top_n_old(n,modelresults,true_sites):
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = get_unique_numbers(modelresult, n)
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

smartcyp_ranks = df_total_cyp2d6['rank_smartcyp'].to_list()
smartcyp_top1_acc = acc_top_n(1,smartcyp_ranks,true_sites_old)
smartcyp_top3_acc = acc_top_n(3,smartcyp_ranks,true_sites_old)
smartcyp_top5_acc = acc_top_n(5,smartcyp_ranks,true_sites_old)
smartcyp_topn = []
smartcyp_topn.append(smartcyp_top1_acc)
smartcyp_topn.append(smartcyp_top3_acc)
smartcyp_topn.append(smartcyp_top5_acc)

somp_ranks = df_total_cyp2d6['rank_somp'].to_list()
somp_top1_acc = acc_top_n(1,somp_ranks,true_sites_old)
somp_top3_acc = acc_top_n(3,somp_ranks,true_sites_old)
somp_top5_acc = acc_top_n(5,somp_ranks,true_sites_old)
somp_topn = []
somp_topn.append(somp_top1_acc)
somp_topn.append(somp_top3_acc)
somp_topn.append(somp_top5_acc)

ourmodel_top1_acc = acc_top_n_old(1,ourmodel_results_new,true_sites_old)
ourmodel_top3_acc = acc_top_n_old(3,ourmodel_results_new,true_sites_old)
ourmodel_top5_acc = acc_top_n_old(5,ourmodel_results_new,true_sites_old)
ourmodel_topn = []
ourmodel_topn.append(ourmodel_top1_acc)
ourmodel_topn.append(ourmodel_top3_acc)
ourmodel_topn.append(ourmodel_top5_acc)

df = pd.DataFrame()
df['top_n'] = ['top1_recall','top3_recall','top5_recall']
df['smartcyp_topn'] = smartcyp_topn
df['somp_topn'] = somp_topn
df['ourmodel_topn'] = ourmodel_topn
df

top 1 accuracy = 0.0
top 3 accuracy = 0.0
top 5 accuracy = 0.1111111111111111
top 1 accuracy = 0.1111111111111111
top 3 accuracy = 0.1111111111111111
top 5 accuracy = 0.3333333333333333
top 1 accuracy = 0.6666666666666666
top 3 accuracy = 0.6666666666666666
top 5 accuracy = 0.8888888888888888


,top_n,smartcyp_topn,somp_topn,ourmodel_topn
0,top1_recall,0.000000,0.111111,0.666667
1,top3_recall,0.000000,0.111111,0.666667
2,top5_recall,0.111111,0.333333,0.888889


In [24]:
df_total_cyp3a4 = df_total[df_total['Enzyme'] == 'CYP3A4']

ourmodel_results_old = df_total_cyp3a4['predict_site_atomidx'].to_list()
ourmodel_results_new = []
for i in ourmodel_results_old:
    list_part = []
    for j in i:
        list_part = list_part + j
    ourmodel_results_new.append(list_part)

true_sites_old = df_total_cyp3a4['site_truth_atomidx'].to_list()

def acc_top_n(n,modelresults,true_sites):
    modelresults = [i[:n] for i in modelresults]
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

def acc_top_n_old(n,modelresults,true_sites):
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = get_unique_numbers(modelresult, n)
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

smartcyp_ranks = df_total_cyp3a4['rank_smartcyp'].to_list()
smartcyp_top1_acc = acc_top_n(1,smartcyp_ranks,true_sites_old)
smartcyp_top3_acc = acc_top_n(3,smartcyp_ranks,true_sites_old)
smartcyp_top5_acc = acc_top_n(5,smartcyp_ranks,true_sites_old)
smartcyp_topn = []
smartcyp_topn.append(smartcyp_top1_acc)
smartcyp_topn.append(smartcyp_top3_acc)
smartcyp_topn.append(smartcyp_top5_acc)

somp_ranks = df_total_cyp3a4['rank_somp'].to_list()
somp_top1_acc = acc_top_n(1,somp_ranks,true_sites_old)
somp_top3_acc = acc_top_n(3,somp_ranks,true_sites_old)
somp_top5_acc = acc_top_n(5,somp_ranks,true_sites_old)
somp_topn = []
somp_topn.append(somp_top1_acc)
somp_topn.append(somp_top3_acc)
somp_topn.append(somp_top5_acc)

ourmodel_top1_acc = acc_top_n_old(1,ourmodel_results_new,true_sites_old)
ourmodel_top3_acc = acc_top_n_old(3,ourmodel_results_new,true_sites_old)
ourmodel_top5_acc = acc_top_n_old(5,ourmodel_results_new,true_sites_old)
ourmodel_topn = []
ourmodel_topn.append(ourmodel_top1_acc)
ourmodel_topn.append(ourmodel_top3_acc)
ourmodel_topn.append(ourmodel_top5_acc)

df = pd.DataFrame()
df['top_n'] = ['top1_recall','top3_recall','top5_recall']
df['smartcyp_topn'] = smartcyp_topn
df['somp_topn'] = somp_topn
df['ourmodel_topn'] = ourmodel_topn
df

top 1 accuracy = 0.06896551724137931
top 3 accuracy = 0.3448275862068966
top 5 accuracy = 0.5172413793103449
top 1 accuracy = 0.1724137931034483
top 3 accuracy = 0.3793103448275862
top 5 accuracy = 0.5172413793103449
top 1 accuracy = 0.5172413793103449
top 3 accuracy = 0.7586206896551724
top 5 accuracy = 0.896551724137931


,top_n,smartcyp_topn,somp_topn,ourmodel_topn
0,top1_recall,0.068966,0.172414,0.517241
1,top3_recall,0.344828,0.379310,0.758621
2,top5_recall,0.517241,0.517241,0.896552
